In [150]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("course_lead_scoring_2026.csv")

In [151]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [152]:
df.shape

(5000, 9)

In [153]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [154]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [155]:
cat_feats = df.select_dtypes(include=['str']).columns
num_feats = df.select_dtypes(exclude=['str']).columns

df[cat_feats] = df[cat_feats].fillna('NA')
df[num_feats] = df[num_feats].fillna(0.0)

In [156]:
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [157]:
unique_vals = df['industry'].value_counts()
print(unique_vals)

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


In [158]:
mode_industry = df['industry'].mode()[0]
print(mode_industry)

technology


In [159]:
corr_matrix = df[num_feats].corr()

feat_pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for feat1, feat2 in feat_pairs:
    r = corr_matrix.loc[feat1, feat2]
    print(f'{feat1} & {feat2}: {r:.4f}')

interaction_count & lead_score: 0.9157
number_of_courses_viewed & lead_score: 0.7572
number_of_courses_viewed & interaction_count: 0.7216
annual_income & interaction_count: 0.1228


In [160]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df, test_size=0.25, random_state=42)

In [161]:
for feat in cat_feats:
    print(f'{feat}: {round(mutual_info_score(df_train.converted, df_train[feat]), 2)}')

lead_source: 0.04
industry: 0.0
employment_status: 0.02
location: 0.0


In [162]:
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [163]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print(f'{accuracy:.2f}')

0.67


In [164]:
feats_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

dv_base = DictVectorizer(sparse=False)
X_train_base = dv_base.fit_transform(train_dicts)
X_val_base = dv_base.transform(val_dicts)

model_base = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_base.fit(X_train_base, y_train)
base_accuracy = accuracy_score(y_val, model_base.predict(X_val_base))

for feat in feats_to_check:
    train_dicts_sub = [{k: v for k, v in d.items() if k != feat} for d in train_dicts]
    val_dicts_sub = [{k: v for k, v in d.items() if k != feat} for d in val_dicts]

    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(train_dicts_sub)
    X_val_sub = dv_sub.transform(val_dicts_sub)

    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)

    sub_accuracy = accuracy_score(y_val, model_sub.predict(X_val_sub))
    diff = base_accuracy - sub_accuracy
    print(f"Drop {feat:<25}: diff = {diff:.5f}")

Drop lead_source              : diff = -0.00320
Drop number_of_courses_viewed : diff = 0.00640
Drop interaction_count        : diff = 0.06560


In [166]:
C_vals = [0.000001, 0.00001, 0.0001, 0.001]

for c in C_vals:
    model_reg = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_reg.fit(X_train_base, y_train)

    y_pred_reg = model_reg.predict(X_val_base)
    acc_reg = accuracy_score(y_val, y_pred_reg)

    print(f'C={c:<10}: Accuracy = {round(acc_reg, 3)}')

C=1e-06     : Accuracy = 0.601
C=1e-05     : Accuracy = 0.601
C=0.0001    : Accuracy = 0.614
C=0.001     : Accuracy = 0.671
